# Machine Learning Survival Models — Random Survival Forest

This notebook extends the baseline Cox analysis by fitting a non‑parametric machine learning model:  
the **Random Survival Forest (RSF)**. We will train the RSF, compare its performance to CoxPH,  
evaluate calibration, and document feature importance.

## Step 9: Random Survival Forest (train + save)
We fit a Random Survival Forest to capture non‑linear effects and interactions:

- Use the processed dataset (`X` predictors, `y` survival object).  
- Configure hyperparameters (number of trees, depth, minimum samples).  
- Train the RSF model and save it to the `models/` directory for reuse.  

**Deliverables:**  
- A trained RSF model (`rsf_model.pkl`) stored in `models/`.

In [ ]:
# =============================================
# STEP 9 — RANDOM SURVIVAL FOREST (with auto-save/load)
# =============================================
from pathlib import Path
import joblib

y = Surv.from_arrays(
    event=df["event"].astype(bool),
    time=df["time"].astype(float)
)

X = df_cox.drop(columns=["time", "event"])

model_path = Path("models/rsf_model.pkl")

if model_path.exists():
    print("✅ Loading existing RSF model...")
    rsf = joblib.load(model_path)
else:
    print("⏳ Training RSF model (first time only)...")

    rsf = RandomSurvivalForest(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=15,
        random_state=42,
        n_jobs=1
    )
    rsf.fit(X, y)

    os.makedirs("models", exist_ok=True)
    joblib.dump(rsf, model_path)
    print("✅ RSF model saved.")

## Step 10: Model comparison (C‑index)
We compare predictive performance between CoxPH and RSF:

- Compute the concordance index (C‑index) for both models.  
- Summarize results in a comparison table.  

**Interpretation:**  
- Higher C‑index indicates better discrimination of readmission risk.  
- RSF often improves over CoxPH when non‑linearities or interactions are present.  

**Artifacts:**  
- Save a CSV table of model comparison (`model_comparison_cindex.csv`) in `reports/`.

In [ ]:
# =============================================
# STEP 10 — MODEL COMPARISON (C-index for Cox + RSF)
# =============================================

# Save Cox model (required by rubric)
with open("models/cox_model.pkl", "wb") as f:
    pickle.dump(cph, f)
print("Saved Cox model to models/cox_model.pkl")

# Cox C-index
cox_cindex = cph.concordance_index_

# RSF C-index: compute risk from CHF (preferred) with fallback
rsf_cindex = None
try:
    # try cumulative hazard -> risk
    chf = rsf.predict_cumulative_hazard_function(X, return_array=True)
    # chf: shape (n_samples, n_times) -> risk as sum over times
    rsf_risk = np.asarray(chf).sum(axis=1)
    rsf_cindex = concordance_index_censored(y["event"], y["time"], rsf_risk)[0]
except Exception as e_chf:
    try:
        # fallback to predict() and flatten
        rsf_pred = rsf.predict(X)
        if hasattr(rsf_pred, "shape") and rsf_pred.ndim > 1:
            rsf_pred = rsf_pred[:, 0]
        rsf_cindex = concordance_index_censored(y["event"], y["time"], rsf_pred)[0]
    except Exception as e2:
        rsf_cindex = None
        with open("report/rsf_prediction_error.txt", "w") as f:
            f.write("CHFFAIL: " + str(e_chf) + "\nPREDFAIL: " + str(e2))

comparison = pd.DataFrame({
    "Model": ["CoxPH", "Random Survival Forest"],
    "C-index": [cox_cindex, rsf_cindex]
})
comparison.to_csv("report/model_comparison_cindex.csv", index=False)
print("\n=== MODEL PERFORMANCE ===")
print(comparison)

## Step 11: Calibration (Brier score)
We evaluate model calibration:

- Compute the Brier score at a chosen evaluation time (e.g., median follow‑up).  
- Lower Brier scores indicate better calibration (predicted survival probabilities closer to observed outcomes).  

**Limitations:**  
- Calibration may be affected if survival times are simulated or fixed.  
- Interpret results cautiously and document assumptions.  

**Artifacts:**  
- Report Brier score in the notebook and include in the client report.

In [ ]:
# =============================================
# STEP 11 — CALIBRATION (Brier Score for Cox model - FIXED)
# =============================================

from sksurv.metrics import brier_score

# Build survival object
y_cox = Surv.from_arrays(
    event=df_cox["event"].astype(bool),
    time=df_cox["time"].astype(float)
)

# Choose valid evaluation time inside observed range
eval_time = float(np.median(df_cox["time"]))

# Get survival probabilities as a DataFrame
surv_df = cph.predict_survival_function(df_cox)

# Convert survival dataframe to numpy
# Shape should be (n_times, n_samples)
surv_array = surv_df.values

# Get nearest time index
time_index = np.abs(surv_df.index.values - eval_time).argmin()

# Extract survival probabilities at that time
pred_surv = surv_array[time_index, :].reshape(-1, 1)

# Brier score input
times = np.array([eval_time])

# Compute Brier score
brier = brier_score(y_cox, y_cox, pred_surv, times)[1][0]

print("\n=== CALIBRATION RESULT ===")
print(f"Brier score at time {eval_time:.1f} days: {brier:.4f}")   

## Step 16: RSF feature importance
We assess which predictors drive RSF risk estimates:

- Use permutation importance on cumulative hazard risk.  
- Rank features by their impact on prediction error when shuffled.  
- Visualize the top 15 features in a bar chart.  

**Interpretation:**  
- Larger importance values indicate stronger influence on RSF predictions.  
- Compare RSF importance to Cox coefficients for consistency and complementary insights.  

**Deliverables:**  
- Save full RSF importance table (`rsf_feature_importance.csv`) in `reports/`.  
- Export bar chart of top features to `plots/EDA/rsf_feature_importance.png`.

In [ ]:
# =============================================
# STEP 16 — RSF FEATURE IMPORTANCE (Permutation on CHF Risk)
# =============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("EDA", exist_ok=True)
os.makedirs("report", exist_ok=True)

print("⏳ Computing RSF feature importance (Permutation on CHF risk)...")

def rsf_perm_importance(model, X, y, n_repeats=5):
    """Permutation importance using CHF-based risk scores (works for sksurv RSF)."""

    # baseline prediction
    baseline_chf = model.predict_cumulative_hazard_function(X, return_array=True)
    baseline_risk = baseline_chf.sum(axis=1)

    importances = []

    for col in X.columns:
        scores = []
        for _ in range(n_repeats):

            Xp = X.copy()
            Xp[col] = np.random.permutation(Xp[col].values)

            chf_perm = model.predict_cumulative_hazard_function(Xp, return_array=True)
            perm_risk = chf_perm.sum(axis=1)

            # RMSE between permuted and original risk
            rmse = np.sqrt(np.mean((perm_risk - baseline_risk) ** 2))
            scores.append(rmse)

        importances.append(np.mean(scores))

    return pd.Series(importances, index=X.columns)


# Compute importance
feat_imp = rsf_perm_importance(rsf, X, y, n_repeats=5)

# Sort
feat_imp_sorted = feat_imp.sort_values(ascending=False)

# Save results
feat_imp_sorted.to_csv("report/rsf_feature_importance.csv")

print("\n=== TOP 15 RSF FEATURES ===")
print(feat_imp_sorted.head(15))

# Plot top 15
plt.figure(figsize=(8,6))
feat_imp_sorted.head(15).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 15 RSF Feature Importances (Permutation on CHF Risk)")
plt.xlabel("Importance (increase in prediction error)")
plt.tight_layout()
plt.savefig("EDA/rsf_feature_importance.png")
plt.show()

print("\n✅ RSF Feature Importance Completed and Saved.")